## Figure 2 - Save transported pleural pressure fields

In [ ]:
# env: notebook
import numpy as np
import os
import pickle

import dolfin

from Reader_MeshDisplacement import MeshDisplacementReader

### Download data

In [ ]:
# TODO Update links to zenodo

In [ ]:
# The transport plans can be computed with the notebook `Fig2b_compute_transport_plans.ipynb` or downloaded from the following link:
!curl -L -o results_POT.zip "https://sdrive.cnrs.fr/s/bMBFcWQF4QmSqDD/download?path=%2F&files=TransportPlansPOT"
!unzip -o results_POT.zip
!rm -f results_POT.zip

### Parameters

In [2]:
pe          = -0.5    # kPa
alpha_lst   = [0.16]
gravity_lst = [1] # [0,1]

rho_solid   = 1e-6
g           = +9.81e3 # mm/s2

### Get pressure data

In [3]:
acquisition_lst = []
acquisition_lst += ["ANTE_SUP2"]
acquisition_lst += ["ANTE_PRO1"]

volunteers_lst  = []
volunteers_lst += ["230526_02BT01"]
volunteers_lst += ["230704_02JY02"]
volunteers_lst += ["230706_02LH03"]
volunteers_lst += ["230728_02CL05"]
volunteers_lst += ["230804_02GA06"]
volunteers_lst += ["230929_02JD07"]
volunteers_lst += ["231013_02BF08"]
volunteers_lst += ["231020_02AR09"]
volunteers_lst += ["231024_02DL10"]
volunteers_lst += ["231027_02AS11"]
volunteers_lst += ["231110_02CC12"]
volunteers_lst += ["231117_02HC13"]
volunteers_lst += ["231124_02VG14"]
volunteers_lst += ["231205_02NS15"]
volunteers_lst += ["231219_02YF16"]
volunteers_lst += ["231222_02OC17"]
volunteers_lst += ["240105_02MD18"]
volunteers_lst += ["240109_02AL19"]
volunteers_lst += ["240119_02JN20"]
volunteers_lst += ["240202_02CH21"]
volunteers_lst += ["240206_02HA04"]
volunteers_lst += ["240524_02TA22"]
volunteers_lst += ["241011_02NB26"]
volunteers_lst += ["241018_02ZT27"]
volunteers_lst += ["241210_02JL29"]
volunteers_lst += ["241217_02JC30"]
volunteers_lst += ["250128_02RP32"]
volunteers_lst += ["250204_02MH35"]
volunteers_lst += ["250207_02SN36"]
volunteers_lst += ["250218_02TG23"]
volunteers_lst += ["250221_02VL38"]
volunteers_lst += ["250304_02WD39"]
volunteers_lst += ["250307_02MC41"]
volunteers_lst += ["250513_02WM46"]
volunteers_lst += ["250909_02LB54"]
volunteers_lst += ["250923_02FM56"]
volunteers_lst += ["250926_02AH57"]
volunteers_lst += ["251014_02RM58"]
volunteers_lst += ["251107_02SL59"]
volunteers_lst += ["251118_02JM60"]
volunteers_lst += ["251128_02LS24"]
volunteers_lst += ["251219_02GD67"]

# Repeated volunteers
# volunteers_lst += ["230718_02HA04"]
# volunteers_lst += ["240827_02TG23"]
# volunteers_lst += ["240903_02LS24"]

region_lst = []
region_lst += ["LL"]
region_lst += ["RL"]

# exclude specific cases
exclude_lst  = []
exclude_lst += ["ANTE_PRO1-231222_02OC17"]
exclude_lst += ["ANTE_PRO2-231222_02OC17"]
exclude_lst += ["POST_PRO1-231222_02OC17"]
exclude_lst += ["ANTE_SUP2-250307_02MC41"]
exclude_lst += ["ANTE_PRO2-250307_02MC41"]
exclude_lst += ["ANTE_SUP2-251014_02RM58"]
exclude_lst += ["ANTE_PRO1-251014_02RM58"]
exclude_lst += ["ANTE_PRO2-251014_02RM58"]
exclude_lst += ["POST_PRO1-251014_02RM58"]
exclude_lst += ["POST_SUP1-251014_02RM58"]
exclude_lst += ["ANTE_PRO1-231013_02BF08-LL"] # mask not correct
exclude_lst += ["ANTE_PRO1-241018_02ZT27-LL"] # mask not correct (back prosthesis)
exclude_lst += ["ANTE_SUP2-241018_02ZT27-LL"] # mask not correct (back prosthesis)

n_phases = 32

In [4]:
results_volunteers = {}
n_sim              = 0

for acquisition in acquisition_lst:
    if acquisition in ["ANTE_SUP1", "ANTE_SUP2", "POST_SUP1"]:
        position = "supine"
    elif acquisition in ["ANTE_PRO1", "ANTE_PRO2", "POST_PRO1"]:
        position = "prone"
    else:
        raise ValueError(f"Unknown position for acquisition {acquisition}")
    
    for volunteer in volunteers_lst:
        if f"{acquisition}-{volunteer}" in exclude_lst:
            continue

        for region in region_lst:
            if f"{acquisition}-{volunteer}-{region}" in exclude_lst:
                continue

            print(f"Processing {acquisition}, {volunteer}, {region} region")

            for gravity_ in gravity_lst:
                if position == "supine":
                    gravity_ = gravity_ * 1
                elif position == "prone":
                    gravity_ = gravity_ * -1

                for alpha_   in alpha_lst:
                    resultsPath = f"./Results_{acquisition}/{volunteer}"

                    # Read unloaded mesh
                    mesh_unloaded = dolfin.Mesh()
                    with dolfin.XDMFFile(f"{resultsPath}/Pleural_pressure_estimation/mesh_unloaded_{region}_alpha{alpha_}_gravity{gravity_}_pe{pe}.xdmf") as xdmf_in:
                        xdmf_in.read(mesh_unloaded)

                    bmesh_unloaded = dolfin.BoundaryMesh(mesh_unloaded, "exterior")

                    # Read pressure values
                    fs_DG0  = dolfin.FunctionSpace(bmesh_unloaded, "DG", 0)
                    p_pl_fn = dolfin.Function(fs_DG0)

                    vfs_CG1 = dolfin.VectorFunctionSpace(bmesh_unloaded, "CG", 1)
                    U_b   = dolfin.Function(vfs_CG1)

                    MDR = MeshDisplacementReader(acquisition, volunteer, region, n_phases)

                    # Read center of mass
                    fs_R           = dolfin.VectorFunctionSpace(bmesh_unloaded, "R", 0)
                    center_of_mass = dolfin.Function(fs_R)

                    h5_results = dolfin.HDF5File(bmesh_unloaded.mpi_comm(), f"{resultsPath}/Pleural_pressure_estimation/data_{region}_alpha{alpha_}_gravity{gravity_}_pe{pe}.h5", "r")

                    pf   = np.zeros((n_phases, bmesh_unloaded.num_cells()))
                    p_pl = np.zeros((n_phases, bmesh_unloaded.num_cells()))

                    for phase_i in range(n_phases):
                        U_b_name = "/U_b/vector_%d"%phase_i
                        h5_results.read(U_b, U_b_name)
                        bmesh_deformed_phase_i = dolfin.BoundaryMesh(mesh_unloaded, "exterior")
                        dolfin.ALE.move(bmesh_deformed_phase_i, U_b)
                        
                        fs_DG0_phase_i         = dolfin.FunctionSpace(bmesh_deformed_phase_i, "DG", 0)
                        def_cells_coords       = fs_DG0_phase_i.tabulate_dof_coordinates()

                        cent_name = "/center_of_mass/vector_%d"%phase_i
                        h5_results.read(center_of_mass, cent_name)
                        # NOTE we are using a center of mass per lung (it may be different in each lung)
                        # print ("Center of mass phase %d: "%phase_i, center_of_mass.vector().get_local())
                        
                        vec_name = "/p_pl/vector_%d"%phase_i
                        h5_results.read(p_pl_fn, vec_name)
                        
                        # timestamp = h5_results.attributes(vec_name)["timestamp"]
                        # print (timestamp)

                        # When saving pressure, negative is inflating the lung (in the outward normal direction)
                        p_pl_i  = p_pl_fn.vector().get_local()
                        x_tilde = def_cells_coords - center_of_mass.vector().get_local().reshape(1,-1)

                        pf_i = p_pl_i.copy()
                        # pf_i -= np.mean(p_pl_i)
                        pf_i -= pe
                        pf_i -= rho_solid * x_tilde.dot(np.array([gravity_*g, 0, 0]))

                        pf[phase_i, :]   = pf_i
                        p_pl[phase_i, :] = p_pl_i

                    with open(f"{resultsPath}/Mesh/Volume_vs_time.pkl", "rb") as f:
                        volume_vol = pickle.load(f)

                    results_volunteers[n_sim] = {"acquisition"  : acquisition,
                                                 "position"     : position,
                                                 "ID"           : volunteer,
                                                 "region"       : region,
                                                 "alpha"        : alpha_,
                                                 "gravity"      : gravity_,
                                                 "pe"           : pe,
                                                 "pf"           : pf,
                                                 "p"            : p_pl,
                                                 "volume_phases": volume_vol
                                                }
                    n_sim += 1

                    h5_results.close()

Processing ANTE_SUP2, 230526_02BT01, LL region
Processing ANTE_SUP2, 230526_02BT01, RL region
Processing ANTE_SUP2, 230704_02JY02, LL region
Processing ANTE_SUP2, 230704_02JY02, RL region
Processing ANTE_SUP2, 230706_02LH03, LL region
Processing ANTE_SUP2, 230706_02LH03, RL region
Processing ANTE_SUP2, 230728_02CL05, LL region
Processing ANTE_SUP2, 230728_02CL05, RL region
Processing ANTE_SUP2, 230804_02GA06, LL region
Processing ANTE_SUP2, 230804_02GA06, RL region
Processing ANTE_SUP2, 230929_02JD07, LL region
Processing ANTE_SUP2, 230929_02JD07, RL region
Processing ANTE_SUP2, 231013_02BF08, LL region
Processing ANTE_SUP2, 231013_02BF08, RL region
Processing ANTE_SUP2, 231020_02AR09, LL region
Processing ANTE_SUP2, 231020_02AR09, RL region
Processing ANTE_SUP2, 231024_02DL10, LL region
Processing ANTE_SUP2, 231024_02DL10, RL region
Processing ANTE_SUP2, 231027_02AS11, LL region
Processing ANTE_SUP2, 231027_02AS11, RL region
Processing ANTE_SUP2, 231110_02CC12, LL region
Processing AN

In [5]:
Simulations_name = []
Simulations_name.append("alpha")
Simulations_name.append(alpha_lst)
Simulations_name.append("pe")
Simulations_name.append(pe)

flat_Simulations_name = [item for sublist in Simulations_name for item in (sublist if isinstance(sublist, list) else [sublist])]
Simulations_filename  = '_'.join([str(x) for x in flat_Simulations_name])

In [6]:
os.makedirs(f"Results_reduced_model/{Simulations_filename}", exist_ok=True)

### Transport pressure fields to the same unloaded (target) mesh

In [7]:
# Target mesh
vol_ID_ref      = "230704_02JY02"
acquisition_ref = "ANTE_SUP2"

for simulation_i in results_volunteers.values():
    if simulation_i["ID"] == vol_ID_ref and simulation_i["acquisition"] == acquisition_ref:
        alpha_ref   = simulation_i["alpha"]
        gravity_ref = simulation_i["gravity"]
        pe_ref      = simulation_i["pe"]
        break

In [ ]:
# Load transport plans and transport the pressure maps to the target mesh
for n_sim, simulation_i in results_volunteers.items():
    pf_acq = simulation_i["pf"]
    p_acq  = simulation_i["p"]

    if simulation_i["ID"] != vol_ID_ref or simulation_i["acquisition"] != acquisition_ref:

        # LL is transported to reference LL and RL to reference RL (reference LL and RL are from the same volutnteer)
        with open(f"TransportPlansPOT/{Simulations_filename}/"
                  f"target_{acquisition_ref}_{vol_ID_ref[-2:]}_{simulation_i['region']}_"
                  f"source_{simulation_i['acquisition']}_{simulation_i['ID'][-2:]}_{simulation_i['region']}.pkl", "rb") as f:
            data        = pickle.load(f)
            T_loaded    = data["T"].toarray()
            log_loaded  = data["log"]
            info_loaded = data["info"]
        
        vol_info = {"acquisition": simulation_i["acquisition"], "ID": simulation_i["ID"], "region": simulation_i["region"],
                    
                    "alpha": simulation_i["alpha"], "gravity": simulation_i["gravity"], "pe": simulation_i["pe"]}
        assert vol_info == info_loaded, "The patient info does not match the loaded info from the transport plans"

        Pf_acq_transported = np.zeros((n_phases, T_loaded.shape[1]))
        P_acq_transported  = np.zeros((n_phases, T_loaded.shape[1]))

        for phase_i in range(n_phases):
            pf_acq_i = np.matmul(T_loaded.T, pf_acq[phase_i,:].reshape(-1,1)) / T_loaded.T.sum(axis=1, keepdims=True)
            p_acq_i  = np.matmul(T_loaded.T, p_acq[phase_i,:].reshape(-1,1)) / T_loaded.T.sum(axis=1, keepdims=True)
            
            Pf_acq_transported[phase_i, :] = pf_acq_i.reshape(1,-1)
            P_acq_transported[phase_i, :]  = p_acq_i.reshape(1,-1)

    else:
        assert simulation_i["ID"] == vol_ID_ref and simulation_i["acquisition"] == acquisition_ref, "The patient info does not match the reference patient info"
        
        Pf_acq_transported = pf_acq
        P_acq_transported  = p_acq
    
    results_volunteers[n_sim]["pf_transported"] = Pf_acq_transported
    results_volunteers[n_sim]["p_transported"]   = P_acq_transported

In [9]:
# Save transported pressure fields in target unloaded mesh
resultsPath              = f"./Results_{acquisition_ref}/{vol_ID_ref}"
fs_DG0_regions_bmesh_ref = {}

for region in region_lst:
    mesh_unloaded = dolfin.Mesh()
    with dolfin.XDMFFile(f"{resultsPath}/Pleural_pressure_estimation/"
                         f"mesh_unloaded_{region}_alpha{alpha_ref}_gravity{gravity_ref}_pe{pe_ref}.xdmf") as xdmf_in:
        xdmf_in.read(mesh_unloaded)

    bmesh_unloaded = dolfin.BoundaryMesh(mesh_unloaded, "exterior")
    fs_DG0         = dolfin.FunctionSpace(bmesh_unloaded, "DG", 0)
    fs_DG0_regions_bmesh_ref[region] = fs_DG0

    with dolfin.XDMFFile(f"Results_reduced_model/{Simulations_filename}/"
                         f"p_transported_target_{acquisition_ref}_{vol_ID_ref[-2:]}_{region}.xdmf") as xdmf:
        xdmf.parameters["flush_output"] = True
        xdmf.parameters["functions_share_mesh"] = True
        xdmf.parameters["rewrite_function_mesh"] = False

        for n_sim, simulation_i in results_volunteers.items():
            if simulation_i["region"] != region:
                continue

            p_transp = dolfin.Function(fs_DG0)
            p_transp.rename(f"p_{simulation_i['acquisition']}_{simulation_i['ID'][-2:]}", f"p_{simulation_i['acquisition']}_{simulation_i['ID'][-2:]}")

            for phase_i in range(n_phases):
                p_transp.vector()[:] = results_volunteers[n_sim]["p_transported"][phase_i, :]
                xdmf.write(p_transp, phase_i)

In [10]:
with open(f"Results_reduced_model/{Simulations_filename}/results_volunteers.pkl", "wb") as f:
    pickle.dump(results_volunteers, f)